# Lab 1 - Fine-tuning Pipeline

Questo notebook copre l'**Exercise 1.3: A Fine-tuning Baseline** e prepara l'**Exercise 2: Pipeline Consolidation**.

L'obiettivo e' passare dalla baseline stabile basata su feature ResNet-18 + SVM a una baseline di fine-tuning semplice, riproducibile e spiegabile. In questa fase alleniamo solo il classificatore finale della ResNet-18, mantenendo congelato il backbone pre-addestrato su ImageNet.

## Richieste dell'esercizio

L'Exercise 1.3 richiede di:

- caricare una ResNet-18 pre-addestrata;
- sostituire il classificatore finale con una nuova testa per le 43 classi GTSRB;
- allenare il modello per alcune epoche;
- monitorare non solo il training loss, ma anche la validation loss su uno split indipendente;
- valutare se il solo fine-tuning della testa finale migliora la baseline precedente.

In questo notebook non inseriamo risultati precompilati: elenchiamo le configurazioni head-only che si possono provare e prepariamo la run principale, lasciando che i risultati vengano prodotti dalla pipeline corrente quando `RUN_TRAINING = True`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dla_lab1.config import experiment_config, load_config
from dla_lab1.data import build_dataloaders, missing_classes_in_split
from dla_lab1.evaluate import classification_metrics, history_to_frame, predict
from dla_lab1.experiments import batch_size_for, run_finetuning
from dla_lab1.models import build_classifier, count_parameters
from dla_lab1.seed import seed_everything
from dla_lab1.train import configure_torch_for_hardware, resolve_device
from dla_lab1.visualize import plot_training_history

config = load_config(ROOT / "config" / "config.yaml")

# Non rilanciamo automaticamente il training lungo.
RUN_TRAINING = False
# Se RUN_TRAINING=True, la run viene salvata anche su Weights & Biases.
ENABLE_WANDB = True
config["wandb"]["enabled"] = ENABLE_WANDB
config["wandb"]["group"] = "exercise_1_3_finetuning_baseline"
# Valore pratico per notebook interattivo su Windows; riduci a 0 se il kernel si blocca.
config["dataset"]["num_workers"] = 4
seed_everything(int(config["project"]["seed"]))

device = resolve_device(config["project"].get("device", "auto"))
configure_torch_for_hardware(device, bool(config["hardware"].get("allow_tf32", True)))

print(f"Project root: {ROOT}")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configurazione della run

Per mantenere un buon rapporto tra costo e prestazioni sul PC disponibile, questa run usa:

- **ResNet-18**, non un modello piu' pesante;
- **backbone congelato**, quindi si addestra solo la c finale;
- **batch size 64**, adatto a una GPU da 4 GB e alla baseline head-only;
- **10 epoche**, sufficienti per una baseline senza rendere il notebook troppo lento;
- **learning rate 0.001**, coerente con lo screening preliminare;
- **AMP attivo** quando si usa CUDA, per velocizzare il training e ridurre memoria.

In [ ]:
EXPERIMENT_NAME = "ex1_3_head_only_adam_ce"
exp_cfg = experiment_config(config, EXPERIMENT_NAME)
training_cfg = exp_cfg["training"]
batch_size = batch_size_for(config, exp_cfg["experiment"]["batch_size_key"])

run_setup = pd.DataFrame([
    ["experiment", EXPERIMENT_NAME],
    ["model", exp_cfg["model"]["name"]],
    ["weights", exp_cfg["model"].get("weights")],
    ["freeze_backbone", exp_cfg["model"].get("freeze_backbone")],
    ["unfreeze_layer4", exp_cfg["model"].get("unfreeze_layer4")],
    ["loss", training_cfg["loss"]],
    ["optimizer", training_cfg["optimizer"]],
    ["learning_rate", training_cfg["learning_rate"]],
    ["batch_size", batch_size],
    ["epochs", training_cfg["epochs"]],
    ["scheduler", training_cfg["scheduler"]],
    ["use_amp", training_cfg["use_amp"] and device.type == "cuda"],
], columns=["parameter", "value"])

run_setup

## Validation split

Il test set ufficiale non viene usato per scegliere gli iperparametri. Per monitorare overfitting e stabilita' della run, dividiamo il training set in train e validation.

Lo split e' costruito per classe e per piccoli gruppi consecutivi di immagini, per ridurre il rischio che immagini quasi identiche finiscano sia in train sia in validation. Qui controlliamo esplicitamente che la validation contenga almeno un elemento per ognuna delle 43 classi.

In [ ]:
loaders = build_dataloaders(
    data_root=ROOT / config["paths"]["data_root"],
    image_size=int(config["dataset"]["image_size"]),
    batch_size=batch_size,
    val_split=float(config["dataset"]["val_split"]),
    track_size=int(config["dataset"]["track_size"]),
    seed=int(config["project"]["seed"]),
    num_workers=int(config["dataset"]["num_workers"]),
    pin_memory=bool(config["dataset"]["pin_memory"]),
    augmentation="none",
)

split_summary = loaders["split_summary"]
missing_val_classes = missing_classes_in_split(split_summary, "val_count")
missing_train_classes = missing_classes_in_split(split_summary, "train_count")

split_check = pd.DataFrame([
    ["train samples", int(split_summary["train_count"].sum())],
    ["validation samples", int(split_summary["val_count"].sum())],
    ["classes in train", int((split_summary["train_count"] > 0).sum())],
    ["classes in validation", int((split_summary["val_count"] > 0).sum())],
    ["missing train classes", missing_train_classes],
    ["missing validation classes", missing_val_classes],
    ["min validation samples per class", int(split_summary["val_count"].min())],
    ["max validation samples per class", int(split_summary["val_count"].max())],
], columns=["check", "value"])

assert not missing_train_classes, f"Classi mancanti nel train split: {missing_train_classes}"
assert not missing_val_classes, f"Classi mancanti nella validation: {missing_val_classes}"

split_check

In [ ]:
split_summary.head(10)

## Prove head-only considerate

Tra le prove considerate ci sono diverse combinazioni di optimizer e loss, sempre con backbone congelato e solo classificatore finale addestrabile. Qui le riportiamo come **piano sperimentale riproducibile**, non come risultati.

Per mantenere il notebook essenziale, la run principale e' `Adam + CrossEntropy`: e' una scelta semplice, spiegabile e adatta come baseline di fine-tuning head-only.


In [ ]:
head_only_experiments = pd.DataFrame([
    ["Adam_CrossEntropy", "CrossEntropy", "Adam", "run principale per la baseline head-only"],
    ["AdamW_CrossEntropy", "CrossEntropy", "AdamW", "alternativa con weight decay decoupled"],
    ["SGD_CrossEntropy", "CrossEntropy", "SGD", "baseline con optimizer classico e momentum"],
    ["Adam_WeightedCrossEntropy", "WeightedCrossEntropy", "Adam", "prova per compensare class imbalance"],
    ["AdamW_WeightedCrossEntropy", "WeightedCrossEntropy", "AdamW", "versione AdamW con pesi di classe"],
    ["SGD_WeightedCrossEntropy", "WeightedCrossEntropy", "SGD", "SGD con pesi di classe"],
    ["Adam_FocalLoss", "FocalLoss", "Adam", "prova focal loss per classi difficili"],
    ["AdamW_FocalLoss", "FocalLoss", "AdamW", "focal loss con AdamW"],
    ["SGD_FocalLoss", "FocalLoss", "SGD", "focal loss con SGD"],
], columns=["Experiment", "Loss", "Optimizer", "Purpose"])

head_only_experiments


Queste prove sono utili per discutere il metodo sperimentale: cambiano loss e optimizer, ma mantengono fisso il vincolo principale dell'Exercise 1.3, cioe' il backbone congelato.

Nel notebook corrente eseguiamo solo la configurazione principale. Se in seguito vogliamo confrontare tutte le varianti, vanno rilanciate nella pipeline corrente e confrontate con lo stesso split train/validation.


## Controllo del modello

Prima di allenare, verifichiamo che la ResNet-18 abbia davvero il backbone congelato e che siano addestrabili solo i parametri della nuova testa finale.

In [ ]:
model = build_classifier(
    model_name=exp_cfg["model"]["name"],
    num_classes=int(config["project"]["num_classes"]),
    weights=exp_cfg["model"].get("weights", "DEFAULT"),
    freeze_backbone=bool(exp_cfg["model"].get("freeze_backbone", True)),
    unfreeze_layer4=bool(exp_cfg["model"].get("unfreeze_layer4", False)),
)

pd.DataFrame([count_parameters(model)])

## Training della baseline head-only

Questa e' la run principale dell'Exercise 1.3. Il training loop non e' scritto manualmente nel notebook: viene richiamato da `src/dla_lab1`, in modo che la pipeline sia riutilizzabile e configurabile.

Per evitare risultati copiati da run precedenti, la cella seguente produce `history` solo se `RUN_TRAINING = True`. Se il training non viene eseguito, non viene mostrata alcuna history artificiale.


In [ ]:
if RUN_TRAINING:
    config["wandb"]["enabled"] = ENABLE_WANDB
    config["wandb"]["group"] = "exercise_1_3_finetuning_baseline"
    result = run_finetuning(config, EXPERIMENT_NAME, root=ROOT)
    history = history_to_frame(result["history"])
    display(history)
else:
    result = None
    history = None
    print("Training non eseguito. Imposta RUN_TRAINING = True per generare la history nel notebook corrente.")


In [ ]:
plot_training_history(history);

## Valutazione su validation

La valutazione viene eseguita solo dopo una run effettiva nel notebook corrente, cosi' report e metriche restano legati all'esecuzione reale.


In [ ]:
if result is not None and history is not None:
    best_row = history.loc[history["val_acc"].idxmax()]
    y_true_val, y_pred_val = predict(result["model"], result["loaders"]["val"], result["device"])
    val_metrics = classification_metrics(y_true_val.numpy(), y_pred_val.numpy())

    validation_summary = pd.DataFrame([
        ["best_epoch", int(best_row["epoch"])],
        ["best_val_accuracy_from_training", float(best_row["val_acc"])],
        ["validation_accuracy_recomputed", float(val_metrics["accuracy"])],
    ], columns=["metric", "value"])

    display(validation_summary)
else:
    val_metrics = None
    print("Validation summary non disponibile: esegui prima il training con RUN_TRAINING = True.")


In [ ]:
if val_metrics is not None:
    print(val_metrics["classification_report"])
else:
    print("Classification report disponibile solo se RUN_TRAINING = True, perche' richiede il modello caricato in memoria.")

## Conclusione Exercise 1.3

Questa sezione rispetta le richieste dell'esercizio: carica una ResNet-18 pre-addestrata, sostituisce la testa finale, prepara uno split train/validation indipendente e usa una pipeline che monitora training e validation.

La configurazione mantenuta come baseline e' `Adam + CrossEntropy` con backbone congelato e sola `fc` addestrabile. Il risultato numerico deve essere prodotto dalla run corrente.

Il passo successivo, coerente con la traccia del professore, sara' sbloccare una parte del backbone, ad esempio `layer4`, e confrontare il miglioramento mantenendo la stessa pipeline configurabile.


## Note sulle funzioni usate

Le funzioni richiamate nel notebook sono commentate direttamente nei file `.py` dentro `src/dla_lab1`. In particolare:

- `build_dataloaders` crea train, validation e test loader;
- `split_class_summary` controlla la distribuzione delle classi nello split;
- `missing_classes_in_split` verifica se mancano classi in train o validation;
- `build_classifier` costruisce la ResNet-18 adattata a 43 classi;
- `run_finetuning` esegue l'intera pipeline di training da configurazione;
- `train_model` gestisce training, validation, checkpoint ed early stopping;
- `history_to_frame` converte la storia del training in tabella;
- `predict` e `classification_metrics` servono per la valutazione finale.